In [ ]:
def ip_to_int(ip: str) -> int:
    """Converts a dotted-decimal IP string to a 32-bit integer."""
    octets = [int(o) for o in ip.split('.')]
    res = 0
    for octet in octets:
        res = (res << 8) + octet
    return res

def cidr_to_range(cidr: str) -> tuple[int, int]:
    """Calculates the start and end integer bounds of a CIDR block."""
    ip_str, prefix = cidr.split('/')
    prefix_len = int(prefix)

    ip_num = ip_to_int(ip_str)

    # Calculate how many total addresses are in this block (2^(32 - prefix))
    num_hosts = 1 << (32 - prefix_len)

    # Find the start of the network (must be a multiple of the block size)
    start = (ip_num // num_hosts) * num_hosts
    end = start + num_hosts - 1

    return start, end

def check_firewall_single_ip(rules: list, ip: str) -> str:
    """
    Checks a single IP against ordered firewall rules.
    Returns 'ALLOW' or 'DENY'.
    """
    target_int = ip_to_int(ip)

    for rule in rules:
        start, end = cidr_to_range(rule['cidr'])

        # If the IP is within this rule's range, return that rule's action
        if start <= target_int <= end:
            return rule['action']

    # If no rules match, default to DENY
    return "DENY"

# --- Example Usage ---
rules = [
    {"action": "DENY", "cidr": "255.0.0.8/29"},      # Covers .8 through .15
    {"action": "ALLOW", "cidr": "117.145.102.64/30"} # Covers .64 through .67
]

print(check_firewall_single_ip(rules, "255.0.0.10"))   # Output: DENY
print(check_firewall_single_ip(rules, "117.145.102.65")) # Output: ALLOW
print(check_firewall_single_ip(rules, "1.1.1.1"))        # Output: DENY (Default)

DENY
ALLOW
DENY


In [ ]:
## Second part:
def check_cidr_block(rules: list, query_cidr: str) -> str:
    """
    Checks if an entire CIDR block is fully allowed by the rules.
    1. Must be covered by ALLOW rules.
    2. Must NOT hit any DENY rules.
    3. Must NOT have any gaps (uncovered IPs).
    """
    q_start, q_end = cidr_to_range(query_cidr)

    # We track the "remaining" parts of the query that need validation.
    # We start with the full query range.
    segments_to_verify = [(q_start, q_end)]

    while segments_to_verify:
        curr_start, curr_end = segments_to_verify.pop(0)
        found_match = False

        for rule in rules:
            r_start, r_end = cidr_to_range(rule['cidr'])

            # Does this rule cover the start of our current segment?
            if r_start <= curr_start <= r_end:
                found_match = True

                # If the first rule we hit for this segment is DENY, fail immediately.
                if rule['action'] == "DENY":
                    return "DENY"

                # If it's an ALLOW rule, check if it covers the whole segment.
                if r_end < curr_end:
                    # It's a partial match! We still need to verify the rest
                    # starting from right after this rule ends.
                    segments_to_verify.append((r_end + 1, curr_end))

                # Segment (or the first part of it) is verified. Move to next.
                break

        # If we went through all rules and didn't find one covering curr_start: GAP.
        if not found_match:
            return "DENY"

    return "ALLOW"

In [ ]:
### Third part:
class SegmentTreeFirewall:
    def __init__(self, rules):
        # 1. Collect all boundary points
        coords = set()
        rule_ranges = []
        for r in rules:
            s, e = cidr_to_range(r['cidr'])
            rule_ranges.append((s, e, r['action']))
            coords.add(s)
            coords.add(e + 1) # Use e+1 to mark the start of the next potential block

        self.sorted_coords = sorted(list(coords))
        # Map a coordinate to its index in the tree
        self.coord_map = {val: i for i, val in enumerate(self.sorted_coords)}

        print(self.coord_map)

        # 2. Initialize tree nodes (each represents a gap between coords)
        # There are len(sorted_coords) - 1 segments
        self.tree = ["DENY"] * (len(self.sorted_coords) - 1)

        # 3. Apply rules in REVERSE order
        # Because we want the FIRST rule to win, we apply the last rules first
        # and let earlier rules overwrite them.
        for s, e, action in reversed(rule_ranges):
            idx_start = self.coord_map[s]
            idx_end = self.coord_map[e + 1]
            for i in range(idx_start, idx_end):
                self.tree[i] = action


    def check_range(self, query_cidr):
        q_s, q_e = cidr_to_range(query_cidr)

        # Ensure query is within our known coordinate bounds
        if q_s < self.sorted_coords[0] or q_e >= self.sorted_coords[-1]:
            # If the query starts before or ends after any rule, check the default
            # but for this logic, we'll treat out-of-bounds as DENY.
            return "DENY"

        # Find which segments our query covers
        import bisect
        # Find start index: the first coord <= q_s
        idx_start = bisect.bisect_right(self.sorted_coords, q_s) - 1
        # Find end index: the first coord > q_e
        idx_end = bisect.bisect_right(self.sorted_coords, q_e) - 1

        # 4. Verification: Every segment in this range must be ALLOW
        for i in range(idx_start, idx_end + 1):
            if self.tree[i] == "DENY":
                return "DENY"

        return "ALLOW"

# --- Usage ---
firewall = SegmentTreeFirewall(rules)
print(firewall.check_range("192.168.1.0/24"))

{1972463168: 0, 1972463172: 1, 4278190088: 2, 4278190096: 3}
DENY
